# Домашка 10 — Классификация дорожных знаков

## Шаг 1. Загружаю данные и смотрю что там

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt

with open('subset_homework.pckl', 'rb') as f:
    data = pickle.load(f)

class_0 = data['class_id_0']
class_1 = data['class_id_1']

print('Класс 0:', len(class_0), 'картинок')
print('Класс 1:', len(class_1), 'картинок')
print('Всего:', len(class_0) + len(class_1))

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(12, 4))

for i in range(6):
    axes[0, i].imshow(class_0[i], cmap='gray')
    axes[0, i].set_title('Класс 0')
    axes[0, i].axis('off')

    axes[1, i].imshow(class_1[i], cmap='gray')
    axes[1, i].set_title('Класс 1')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

В классе 0 — 2220 картинок, в классе 1 — 2250. Примерно поровну.

## Подготовка данных

In [ ]:
X_0 = np.array(class_0, dtype=np.float32)
X_1 = np.array(class_1, dtype=np.float32)

X = np.concatenate([X_0, X_1], axis=0)
y = np.concatenate([np.zeros(len(X_0)), np.ones(len(X_1))])

# делю на 255 чтобы значения были от 0 до 1
X = X / 255.0

# превращаю картинки в вектора
X = X.reshape(X.shape[0], -1)

print('X:', X.shape)
print('y:', y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print('Обучающая выборка:', X_train.shape[0])
print('Тестовая выборка:', X_test.shape[0])

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)

## Шаг 2. Один нейрон

In [ ]:
# функция для обучения, чтобы не писать одно и то же дважды
def train_model(model, epochs=30, lr=0.001):
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    losses = []
    accs = []

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for xb, yb in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        model.eval()
        with torch.no_grad():
            pred = (model(X_test_t) > 0.5).float()
            acc = (pred == y_test_t).float().mean().item()

        losses.append(epoch_loss / len(train_loader))
        accs.append(acc)

        if (epoch + 1) % 10 == 0:
            print(f'Эпоха {epoch+1}: loss={losses[-1]:.4f}, accuracy={acc:.4f}')

    return losses, accs

In [ ]:
# один нейрон — по сути логистическая регрессия
model1 = nn.Sequential(
    nn.Linear(784, 1),
    nn.Sigmoid()
)

losses_1, accs_1 = train_model(model1)
print(f'\nЛучшая точность: {max(accs_1):.4f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
ax1.plot(losses_1)
ax1.set_title('Loss')
ax2.plot(accs_1)
ax2.set_title('Accuracy')
plt.tight_layout()
plt.show()

С одним нейроном получилось около 88%. Неплохо, но попробую лучше.

## Шаг 3. Добавляю больше слоёв

In [ ]:
model2 = nn.Sequential(
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 1),
    nn.Sigmoid()
)

losses_2, accs_2 = train_model(model2)
print(f'\nЛучшая точность: {max(accs_2):.4f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))

ax1.plot(losses_1, label='1 нейрон')
ax1.plot(losses_2, label='3 слоя')
ax1.set_title('Loss')
ax1.legend()

ax2.plot(accs_1, label='1 нейрон')
ax2.plot(accs_2, label='3 слоя')
ax2.set_title('Accuracy')
ax2.legend()

plt.tight_layout()
plt.show()

## Итог

- Один нейрон: ~88%
- Сеть с тремя слоями (128-64-1): ~96%

Добавление скрытых слоёв сильно улучшило результат!